# Построение векторной базы для multilingual-e5-large (JSON-чанки + Markdown)

Ноутбук может читать два источника данных:
1. `data/json_chunks/*.json` (уже отчанкированные тексты)
2. `data/*.md` (будут нарезаны на чанки)

Важно для e5: документы кодируются как `passage: ...`, запросы как `query: ...`.


In [ ]:
from pathlib import Path
import json

import faiss
import numpy as np
import torch
from sentence_transformers import SentenceTransformer


In [ ]:
DATA_JSON_DIR = Path("data/json_chunks")
DATA_MD_DIR = Path("data")
VECTOR_DB_DIR = Path("vector_db_e5_large")

# TODO (изолированная среда): вставьте путь к локальной модели e5-large
# Пример: /models/multilingual-e5-large
EMBEDDING_MODEL_PATH = "/home/vladislav/models/multilingual-e5-large"
PREFERRED_EMBEDDING_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Источники данных
USE_JSON_CHUNKS = True
USE_MARKDOWN = True

# Настройки чанкинга для markdown
CHUNK_SIZE = 800
CHUNK_OVERLAP = 200

# Жестко ограничиваемся одним потоком
torch.set_num_threads(1)
if hasattr(torch, "set_num_interop_threads"):
    torch.set_num_interop_threads(1)
faiss.omp_set_num_threads(1)

if CHUNK_OVERLAP >= CHUNK_SIZE:
    raise ValueError("CHUNK_OVERLAP должен быть меньше CHUNK_SIZE")

def resolve_embedding_device(preferred_device: str) -> str:
    device = (preferred_device or "cpu").strip().lower()
    if device != "cuda":
        return "cpu"
    if not torch.cuda.is_available():
        return "cpu"
    try:
        _ = torch.zeros(1, device="cuda")
        return "cuda"
    except Exception as exc:
        print(f"CUDA недоступна для текущего torch/GPU ({exc}). Переключаемся на CPU.")
        return "cpu"

def _to_int(value, fallback: int) -> int:
    try:
        return int(value)
    except Exception:
        return fallback

def _extract_json_chunks(payload, path: Path) -> list[dict]:
    rows: list[dict] = []

    if isinstance(payload, list):
        items = payload
        base_source = path.stem
    elif isinstance(payload, dict):
        chunks = payload.get("chunks")
        if isinstance(chunks, list):
            items = chunks
        else:
            items = [payload]
        base_source = str(payload.get("source") or payload.get("doc_id") or path.stem)
    else:
        return rows

    for i, item in enumerate(items):
        if isinstance(item, str):
            text = item.strip()
            source = base_source
            chunk_id = i
        elif isinstance(item, dict):
            text = str(item.get("text") or item.get("chunk") or item.get("content") or "").strip()
            source = str(item.get("source") or item.get("doc_id") or base_source)
            chunk_id = _to_int(item.get("chunk_id", i), i)
        else:
            continue

        if text:
            rows.append({"text": text, "source": source, "chunk_id": chunk_id})

    return rows

def read_chunked_json_documents(data_json_dir: Path) -> list[dict]:
    rows: list[dict] = []
    if not data_json_dir.exists():
        return rows
    for path in sorted(data_json_dir.glob("*.json")):
        payload = json.loads(path.read_text(encoding="utf-8"))
        rows.extend(_extract_json_chunks(payload, path))
    return rows

def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    step = chunk_size - overlap
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end >= len(text):
            break
        start += step
    return chunks

def read_markdown_documents(data_md_dir: Path) -> list[dict]:
    rows: list[dict] = []
    if not data_md_dir.exists():
        return rows
    for path in sorted(data_md_dir.glob("*.md")):
        text = path.read_text(encoding="utf-8").strip()
        if not text:
            continue
        for i, piece in enumerate(chunk_text(text)):
            rows.append({"text": piece, "source": str(path), "chunk_id": i})
    return rows

EMBEDDING_DEVICE = resolve_embedding_device(PREFERRED_EMBEDDING_DEVICE)


In [ ]:
if not USE_JSON_CHUNKS and not USE_MARKDOWN:
    raise ValueError("Нужно включить хотя бы один источник: USE_JSON_CHUNKS или USE_MARKDOWN")

chunks: list[dict] = []
if USE_JSON_CHUNKS:
    chunks.extend(read_chunked_json_documents(DATA_JSON_DIR))
if USE_MARKDOWN:
    chunks.extend(read_markdown_documents(DATA_MD_DIR))

if not chunks:
    raise FileNotFoundError(
        f"Не найдено данных. Проверьте JSON в {DATA_JSON_DIR.resolve()} и Markdown в {DATA_MD_DIR.resolve()}"
    )

texts = [item["text"] for item in chunks]
e5_passages = [f"passage: {text}" for text in texts]

print(f"Используется устройство для эмбеддингов: {EMBEDDING_DEVICE}")
embedder = SentenceTransformer(EMBEDDING_MODEL_PATH, device=EMBEDDING_DEVICE)

try:
    vectors = embedder.encode(
        e5_passages,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True,
        device=EMBEDDING_DEVICE,
    ).astype(np.float32)
except Exception as exc:
    if EMBEDDING_DEVICE != "cuda":
        raise
    print(f"Ошибка CUDA на encode ({exc}). Переключаемся на CPU.")
    EMBEDDING_DEVICE = "cpu"
    embedder = SentenceTransformer(EMBEDDING_MODEL_PATH, device=EMBEDDING_DEVICE)
    vectors = embedder.encode(
        e5_passages,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True,
        device=EMBEDDING_DEVICE,
    ).astype(np.float32)

index = faiss.IndexFlatIP(vectors.shape[1])
index.add(vectors)

VECTOR_DB_DIR.mkdir(parents=True, exist_ok=True)
faiss.write_index(index, str(VECTOR_DB_DIR / "index.faiss"))

docs_payload = chunks
(VECTOR_DB_DIR / "docs.json").write_text(
    json.dumps(docs_payload, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

sources_count = len({item['source'] for item in chunks})
print(f"Источников: {sources_count}")
print(f"Чанков: {len(chunks)}")
print(f"Размерность эмбеддинга: {vectors.shape[1]}")
print(f"Сохранено: {(VECTOR_DB_DIR / 'index.faiss').resolve()}")
print(f"Сохранено: {(VECTOR_DB_DIR / 'docs.json').resolve()}")


In [ ]:
# Мини-проверка поиска по построенному индексу
query = "Какие три приоритета выделены в стратегии до 2040 года?"
query_vec = embedder.encode(
    [f"query: {query}"],
    normalize_embeddings=True,
    convert_to_numpy=True,
    device=EMBEDDING_DEVICE,
).astype(np.float32)
scores, idxs = index.search(query_vec, k=3)

for rank, (score, idx) in enumerate(zip(scores[0], idxs[0]), start=1):
    if idx < 0:
        continue
    chunk = docs_payload[idx]
    preview = chunk["text"][:220].replace("\n", " ")
    print(f"{rank}. score={score:.4f} | source={chunk['source']} | chunk_id={chunk['chunk_id']}")
    print(preview)
    print('-' * 80)
